# Smoke Test

Run this notebook from the repository root. It checks imports, synthetic data generation, core conformal methods, and a tiny end-to-end synthetic experiment.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "utility").exists():
    raise RuntimeError(f"Run this notebook from the repository root, not {repo_root}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")

In [ ]:
import numpy as np
import pandas as pd

from utility.data_generator import make_multitarget_regression
from utility.rectangle import Rectangle
from utility.unscaled import unscaled_prediction, bonferroni_prediction
from utility.data_splitting import (
    naive_prediction,
    data_splitting_standardized_prediction,
    data_spliting_CHR_prediction,
)
from utility.copula import empirical_copula_prediction
from utility.res_rescaled import check_coverage_rate, standardized_prediction

print("Core imports: OK")

In [ ]:
noise_cases = [
    ("Gaussian", {}),
    ("Laplace", {}),
    ("Gamma", {}),
    ("Mixed", {}),
    ("Cauchy", {}),
    ("t", {"df": 3}),
]

for noise_type, kwargs in noise_cases:
    X, y, coef = make_multitarget_regression(
        n_samples=24,
        n_features=6,
        n_informative=4,
        n_targets=3,
        noise_type=noise_type,
        noise_list=[1.0, 2.0, 3.0],
        random_state=123,
        **kwargs,
    )
    assert X.shape == (24, 6)
    assert y.shape == (24, 3)
    assert len(coef) == 3

print("Data generator smoke test: OK")

In [ ]:
rng = np.random.default_rng(42)
scores_cal = np.abs(rng.normal(loc=[1.0, 2.0], scale=[0.5, 1.0], size=(40, 2)))
scores_test = np.abs(rng.normal(loc=[1.0, 2.0], scale=[0.5, 1.0], size=(80, 2)))

methods = {
    "TSCP_R": lambda s: standardized_prediction(s, alpha=0.2, short_cut=True),
    "TSCP_GWC": lambda s: standardized_prediction(s, alpha=0.2, method="GWC", short_cut=True),
    "Unscaled": lambda s: unscaled_prediction(s, alpha=0.2),
    "Bonferroni": lambda s: bonferroni_prediction(s, alpha=0.2),
    "Naive": lambda s: naive_prediction(s, alpha=0.2),
    "TSCP_S": lambda s: data_splitting_standardized_prediction(s, alpha=0.2),
    "Point_CHR": lambda s: data_spliting_CHR_prediction(s, alpha=0.2),
    "Empirical_copula": lambda s: empirical_copula_prediction(s, alpha=0.2),
}

rows = []
for name, fn in methods.items():
    region = fn(scores_cal)
    assert isinstance(region, Rectangle)
    assert region.upper.shape == (2,)
    coverage = check_coverage_rate(scores_test, region)
    assert 0.0 <= coverage <= 1.0
    rows.append({
        "method": name,
        "coverage": coverage,
        "volume": region.volume(),
        "max_length": region.length_along_dimensions().max(),
    })

pd.DataFrame(rows)

In [ ]:
from utility.exps import run_abs_res_synthetic_experiment

result = run_abs_res_synthetic_experiment(
    dim_list=[10],
    sample_list=[100],
    alpha_list=[0.1],
    noise_type="Gaussian",
    trials=200,
    methods=["TSCP_R", "Unscaled", "Bonferroni"],
    n_train=int(0.8 * (8000)), n_test=(8000) - int(0.8 * (8000)),
    n_features=10,
    n_informative=10,
    oracle_n_samples=200,
)

assert not result.trial_results.empty
assert not result.summary_results.empty
assert set(result.summary_results["method"]) == {"TSCP_R", "Unscaled", "Bonferroni"}

result.summary_results

Optional: once the quick checks above pass, you can run a slightly larger case by increasing `trials`, `n_train`, `n_test`, dimensions, or the method list in the previous cell.